# 3 - Match NYSDOT AADT to LION and calculate exposure

NYSDOT and LION are separate roadway networks, so this is a spatial correspondence rather than an ID join. keep distance and street-name agreement as match-quality diagnostics.

In [ ]:
from pathlib import Path
import sys

# Make imports work whether Jupyter starts in the repo root or notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CACHE_DIR = ROOT / "data" / "cache"
OUTPUT_DIR = ROOT / "data" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_START = "2021-01-01"
ANALYSIS_END = "2025-12-31"
REFRESH = False

print("Repo root:", ROOT)
print("Analysis period:", ANALYSIS_START, "to", ANALYSIS_END)
print("Force API refresh:", REFRESH)


In [ ]:
from src.api import load_cache, save_cache
from src.crashes import fetch_crashes, crashes_to_geodataframe
from src.lion import fetch_lion, prepare_lion_segments
from src.aadt import fetch_aadt
from src.joins import (
    join_crashes_to_lion, aggregate_crashes_by_segment, attach_crash_metrics,
    join_aadt_to_lion, add_exposure_metrics,
)


In [ ]:
# Reuse notebook 02 output if available; otherwise rebuild it.
segments_with_crashes = load_cache("segments_with_crashes", cache_dir=CACHE_DIR)

if segments_with_crashes is None:
    crashes = fetch_crashes(ANALYSIS_START, ANALYSIS_END, refresh=REFRESH, cache_dir=CACHE_DIR)
    lion_segments = prepare_lion_segments(fetch_lion(refresh=REFRESH, cache_dir=CACHE_DIR))
    crash_lion = join_crashes_to_lion(crashes_to_geodataframe(crashes), lion_segments)
    segment_crashes = aggregate_crashes_by_segment(crash_lion)
    segments_with_crashes = attach_crash_metrics(lion_segments, segment_crashes)

print(f"Segments: {len(segments_with_crashes):,}")


In [ ]:
aadt = fetch_aadt(refresh=REFRESH, cache_dir=CACHE_DIR)

segments_with_aadt = join_aadt_to_lion(
    segments_with_crashes,
    aadt,
    max_distance_ft=100,
)

print(segments_with_aadt["aadt_match_confidence"].value_counts(dropna=False))


In [ ]:
qa_cols = [c for c in [
    "segmentid", "street", "roadway_name", "aadt", "aadt_distance_ft",
    "aadt_name_match", "aadt_match_confidence", "calculation_year", "year_last_act"
] if c in segments_with_aadt.columns]

display(
    segments_with_aadt[qa_cols]
    .sort_values(["aadt_match_confidence", "aadt_distance_ft"], ascending=[True, False])
    .head(30)
)


## Add VMT-based rates

`study-period VMT = AADT × study days × segment length (miles)`

Rates are expressed per 100 million vehicle-miles traveled. This adjusts for motor vehicles only!; does not account for pedestrian/cyclist exposure.

In [ ]:
segments_exposure = add_exposure_metrics(
    segments_with_aadt,
    start_date=ANALYSIS_START,
    end_date=ANALYSIS_END,
)

print(f"Rate-eligible (high/medium AADT match): {segments_exposure['rate_eligible'].sum():,}")
display(
    segments_exposure.loc[segments_exposure["rate_eligible"]]
    .sort_values("injuries_per_100m_vmt", ascending=False)[
        [c for c in [
            "segmentid", "street", "people_injured", "aadt", "length_miles",
            "injuries_per_100m_vmt", "aadt_match_confidence"
        ] if c in segments_exposure.columns]
    ].head(25)
)


In [ ]:
save_cache(
    segments_exposure,
    "segments_with_exposure",
    cache_dir=CACHE_DIR,
    metadata={"analysis_start": ANALYSIS_START, "analysis_end": ANALYSIS_END},
)
print("Saved data/cache/segments_with_exposure.pkl")
